In [1]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split

In [3]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [4]:
df = pd.read_csv(r"C:\Users\adamc\Desktop\ALL FOLDERS & FILES\MSBA Summer 26\GBA 6410\Project\Data Cleaning Pipelines - OFFICIAL\comments_clean_final.csv", encoding="utf-8-sig")
df = df.dropna(subset=["clean_body_nostop"]).reset_index(drop=True)
print("Rows:", f"{len(df):,}")  

Rows: 498,147


In [6]:
# Vectorizer + 80/20 split

domain_stopwords = ENGLISH_STOP_WORDS.union(
    ["la", "lol", "post", "yeah", "first", "day", "today", "fuck", "shit"]
)
count_vectorizer = CountVectorizer(
    max_df=0.90, min_df=10,
    stop_words=list(domain_stopwords),
    token_pattern=r"\b[a-zA-Z]{2,}\b",
)
text_train, text_test = train_test_split(
    df["clean_body_nostop"], test_size=0.20, random_state=RANDOM_STATE
)
X_counts_train = count_vectorizer.fit_transform(text_train)
print("Train:", X_counts_train.shape)


Train: (398517, 16291)


In [7]:
# final model K=5

lda = LatentDirichletAllocation(
    n_components=5, learning_method="online", batch_size=256,
    n_jobs=1, random_state=RANDOM_STATE,
)
lda.fit(X_counts_train) 

,n_components,5
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'online'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,256
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [8]:
# Assign topics to every document

doc_topic_lda = lda.transform(count_vectorizer.transform(df["clean_body_nostop"]))
df["topic_id"]   = doc_topic_lda.argmax(axis=1)
df["topic_prob"] = doc_topic_lda.max(axis=1)

labels = {
    0: "Housing Affordability and Urban Issues",
    1: "Neighborhoods and Living Conditions",
    2: "General Discussions and Everyday Experiences",
    3: "Politics and Public Policy",
    4: "Community and Local Events",
}
df["topic_label"] = df["topic_id"].map(labels)

In [9]:
# Check, save results as parquet to local folder

print("\nTopic sizes:")
print(df["topic_label"].value_counts())

print("\nAssignment confidence:")
print(df["topic_prob"].describe())
for t in [0.40, 0.50, 0.60]:
    print(f"  below {t:.2f}: {(df['topic_prob'] < t).mean():.1%}")

df[["comment_id", "topic_id", "topic_label", "topic_prob"]].to_parquet(
    "topic_assignments.parquet", index=False
)


Topic sizes:
topic_label
Housing Affordability and Urban Issues          126908
Neighborhoods and Living Conditions             104443
Politics and Public Policy                       94345
General Discussions and Everyday Experiences     91442
Community and Local Events                       81009
Name: count, dtype: int64

Assignment confidence:
count    498147.000000
mean          0.503482
std           0.142136
min           0.200000
25%           0.399995
50%           0.479543
75%           0.596736
max           0.970370
Name: topic_prob, dtype: float64
  below 0.40: 25.6%
  below 0.50: 54.2%
  below 0.60: 78.0%


In [10]:
# representative documents for the mislabeled topic

for t in [4, 2]:
    print(f"\n===== TOPIC {t}: {labels[t]} =====")
    top = np.argsort(doc_topic_lda[:, t])[::-1][:8]
    for i in top:
        print(f"  [{doc_topic_lda[i, t]:.2f}] {df['body'].iloc[i][:160]}")


===== TOPIC 4: Community and Local Events =====
  [0.97] What about Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan and Morgan
  [0.93] Whoa whoa whoa buddy calm down with this logic and reasoning - this is reddit and people need to fear monger and virtue signal. 
  [0.93] It's a fucking event, not an invasion. When in the past have the military been necessary for a sports riot? Protest? Terrorist takeover of the jumbotron?
  [0.93] I carry a knife on all walks in case I encounter an aggressive off-leash dog. Not gonna be mauled without killing the mauler.
  [0.93] Omg please ask them WHY. I see them all over LA and can’t for the life of me figure out what these digi-plates are bringing to my life to justify their subscrip
  [0.93] Wow, you're an awful person. I'm glad I don't know you and bet you've been shunned by all the wonderful people in your life. And for good reason too
  [0.93] I agree 